<a href="https://colab.research.google.com/github/devikamanoj1491/Interest_Profiler/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [46]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
flex is already the newest version (2.6.4-8.2build1).
bison is already the newest version (2:3.8.2+dfsg-1ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


## Lex only

In [47]:
#@title Writing Lex program
%%writefile program.l

%{
  #include <stdio.h>
    int n_statements = 0;
    int n_relational = 0;
    int n_keywords = 0;
    int n_integers = 0;
    int n_assign = 0;
    int n_identifiers = 0;
%}
%%
"int"|"char"|"short"|"long"|"float"|"double"|"include"|"if"|"else"|"for"|"do"|"while"|"switch"|"case"|"break"|"return"|"continue"   {n_keywords++;}
"=="|"<="|">="|"<"|">"|"!="     {n_relational++;}
[a-zA-Z_][a-zA-Z0-9_]*          {n_identifiers++;}
[0-9]+                          {n_integers++;}
";"                             {n_statements++;}
"+="|"-="|"*="|"/="|"="         {n_assign++;}
%%
int main(int argc, char *argv[]){
  if(argc>1){
    FILE *file  = fopen(argv[1],"r");
  if(!file){
    perror("File cannot be opened");
    return 1;
  }yyin = file;
  }


yylex();
printf("Number of statements:%d\n", n_statements);
printf("Number of assignment operators:%d\n", n_assign);
printf("Number of relational operators:%d\n", n_relational);
printf("Number of keywords:%d\n", n_keywords);
printf("Number of integers:%d\n", n_integers);
printf("Number of identifiers:%d\n", n_identifiers);
return 0;
}
int yywrap(){
  return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [48]:
%%writefile program.txt

hello I am devika;
i am >= 22 years old;
123444;
20+=1;
mellie;

Overwriting program.txt


In [49]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt


   
     



Number of statements:5
Number of assignment operators:1
Number of relational operators:1
Number of keywords:0
Number of integers:4
Number of identifiers:9


## Lex and Yacc combined

In [50]:
#@title Writing YACC program
%%writefile program.y

%{
  #include <stdio.h>
  #include <stdlib.h>
  int yylex();
  int yyerror(char *s);
%}
%token FOR NUM ID RELOP INC_DEC
%%
input:   stmt  '\n'  {printf("Valid");exit(0);}
;
stmt: FOR '(' stmt_list ')' '{' body '}'
;
stmt_list:  assign ';' cond ';' inc
;
assign: ID '=' NUM | ID '=' ID
;
body: ID '=' ID ';'| ID RELOP NUM ';'| stmt | ID '=' NUM ';'
;
cond: ID RELOP NUM
;
inc: ID INC_DEC
%%
int main(){
  printf("Enter the statement: ");
  yyparse();
  return 0;
}
int yyerror(char *s){
  printf("Invalid statement\n");
  return 1;
}

Overwriting program.y


In [51]:
#@title Writing Lex program
%%writefile program.l

%{
  #include "y.tab.h"
%}
%%
"for"    {return FOR;}
[a-zA-Z_][a-zA-Z0-9_]*         {return ID;}
[0-9]+   {return NUM;}
"<="|">="|"<"|">"|"!="         {return RELOP;}
"++"|"--"     {return INC_DEC;}
"="       {return yytext[0];}
"("       {return yytext[0];}
")"       {return yytext[0];}
";"       {return yytext[0];}
"{"       {return yytext[0];}
"}"       {return yytext[0];}
[\n]      {return '\n';}
[ \t]     ;
.         {return yytext[0];}
%%
int yywrap(){
  return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [52]:
%%writefile program.txt

This is a sample file.

Overwriting program.txt


In [53]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out

Enter the statement: for(i=0;i<10;i++){a=10;}
Valid